In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent:
    if (project_root / 'src').exists():
        break
    project_root = project_root.parent
else:
    raise RuntimeError('Unable to locate project root containing src/')

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


# RPFR Walkthrough (CCSD(T) Example)

Demonstrate how to pair published constants for the reference isotopologue with CCSD(T) spectroscopic constants parsed from a `Fit_1D` output, then evaluate the RPFR for the substitution.

In [4]:
from richet_rpfr import (
    load_molecular_constants_from_excel,
    load_molecular_constants_from_fit_output,
    PartitionFunctionCalculator,
)
import pandas as pd

raw_excel = project_root / 'data' / 'processed' / 'spreadsheets' / 'Richet - RPFR & mol constans - diatoms.xlsx'
constants = load_molecular_constants_from_excel(raw_excel, sheet_name='Diatoms (Table 5)')

NSI_CO = constants['12C16O']
ssi_path = project_root / 'data' / 'raw' / 'CCSD(T)' / '14C18O.7pointfit.out'
SSI_CO = load_molecular_constants_from_fit_output(ssi_path)

T_0C = 273.15

calc = PartitionFunctionCalculator(temperature=T_0C, NSI=NSI_CO, SSI=SSI_CO)
calc.calculate_all()
calc.print_contribution_table()

contributions = pd.Series({
    'Translational': calc.Q_ratio_trans,
    'ZPE_G0': calc.Q_ratio_ZPE_G0,
    'ZPE_harmonic': calc.Q_ratio_ZPE_harmonic,
    'ZPE_anharmonic': calc.Q_ratio_ZPE_anharmonic,
    'ZPE_total': calc.Q_ratio_ZPE_total,
    'Excited (harmonic)': calc.Q_ratio_excited_harmonic,
    'Excited (anharmonic)': calc.Q_ratio_excited_anharmonic,
    'Rotational linear': calc.Q_ratio_rotational_linear,
    'Rotational linear (corr.)': calc.Q_ratio_rotational_linear_w_corr,
    'Rotational nonlinear (corr.)': calc.Q_ratio_rotational_nonlinear_corr,
    'Rotational diatomic': calc.Q_ratio_rotational_diatomic,
    'Rotational-vibrational': calc.Q_ratio_rotational_vibrational,
    'Total Q': calc.Q_tot,
    'RPFR': calc.RPFR,
})
contributions


Contribution                  | Value
------------------------------+------
Translational                 | 1.222236e+00
ZPE (G0)                      | 1.000000e+00
ZPE (harmonic)                | 1.442896e+00
ZPE (anharmonic)              | 9.983422e-01
ZPE (total)                   | 1.440504e+00
Excited (harmonic)            | 1.000012e+00
Excited (anharmonic)          | 9.999987e-01
Rotational (linear)           | 1.141736e+00
Rotational (linear, corr.)    | 1.141257e+00
Rotational (nonlinear, corr.) | 1.000000e+00
Rotational (diatomic)         | 1.141257e+00
Rotational-vibrational        | 1.000000e+00
Q total                       | 2.009360e+00
RPFR                          | 2.009360e+00

Notes:
- 12C16O: missing delta constants for rotational-vibrational contribution; assumed 0.0 and contribution set to 1.0.


Translational                   1.222236
ZPE_G0                          1.000000
ZPE_harmonic                    1.442896
ZPE_anharmonic                  0.998342
ZPE_total                       1.440504
Excited (harmonic)              1.000012
Excited (anharmonic)            0.999999
Rotational linear               1.141736
Rotational linear (corr.)       1.141257
Rotational nonlinear (corr.)    1.000000
Rotational diatomic             1.141257
Rotational-vibrational          1.000000
Total Q                         2.009360
RPFR                            2.009360
dtype: float64